# Create Embedding Vectors for Documents


## Step 1:  Extracting documents



### Loading External Environment

In [61]:
# Loading external env

%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

from typing import List, Dict
from glob import glob
from datetime import datetime
from tqdm import tqdm
from time import sleep


import os
import sys
import logging
import json
import uuid
import chromadb


sys.path.append('../05_src/')

CHROMA_URL = os.getenv("CHROMA_URL")
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')



The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


### Extracting Single File Text Stored in a List

This idea here is to get all the productId from the Statistics Canada website. The JSON response will contain the productId as well as the product description. By embedding the product description, it becomes possible to use RAG to identify which product to use for graph creation. 

#### Get Data

In [ ]:
import requests

endpoint_api_url="https://www150.statcan.gc.ca/t1/wds/rest/getAllCubesListLite"

class Product:

    def __init__(self, 
                 productId, 
                 cansimId, 
                 cubeTitleEn, 
                 cubeTitleFr, 
                 cubeStartDate,
                 cubeEndDate, 
                 releaseTime, 
                 archived,
                 subjectCode,
                 surveyCode, 
                 frequencyCode,
                 corrections, 
                 issueDate):
        self.productId = productId
        self.cansimId = cansimId
        self.cubeTitleEn = cubeTitleEn
        self.cubeTitleFr = cubeTitleFr
        self.cubeStartDate = cubeStartDate
        self.cubeEndDate = cubeEndDate
        self.releaseTime = releaseTime
        self.archived = archived
        self.subjectCode = subjectCode
        self.surveyCode = surveyCode
        self.frequencyCode = frequencyCode
        self.corrections = corrections
        self.issueDate = issueDate


def get_list_of_products() -> List[Product]:
    try:
        response = requests.get(endpoint_api_url)
        formatted_response = response.json()

        return formatted_response
    except:
        logging.error("The productIds could not be fetch.")
        return []


all_products = get_list_of_products()


#### Format for Batch Embeddings

In [ ]:

def format_products(products: List[Product]) -> List[Dict]:

    formatted = []

    for product in products:

        productId = product.get("productId")
        metadata = {
            "cansimId": product.get("cansimId"),
            "cubeStartDate": product.get("cubeStartDate"),
            "cubeEndDate": product.get("cubeEndDate"),
            "releaseTime": product.get("releaseTime"),
            "subjectCode": product.get("subjectCode"),
            "frequencyCode": product.get("cansimId"),
            "issueDate": product.get("issueDate"),
            "createdAt": datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ")
                            }
        content = product.get("cubeTitleEn")

        formatted_product={
            "productId": productId,
            "metadata": metadata,
            "content": content
        }

        formatted.append(formatted_product)

    
    return formatted



structured_products = format_products(all_products)

    

#### Batching

In [ ]:
def create_single_batch_file(output_file_template):

    # Create batches

    batch_size = 150

    batches = [structured_products[i: i+ batch_size] for i in range(0, len(structured_products), batch_size) ]
        
    
    for index, batch in enumerate(batches):

        with open(f"../assignment_2/documents/{output_file_template}-{index + 1}.jsonl", 'w') as outfile:
                
            for product in batch:
                out_dict = {
                        "custom_id": str(product["productId"]), 
                        "method": "POST", 
                        "url": "/v1/embeddings", 
                        "body": {
                            "model": "text-embedding-3-small", 
                            "input": product["content"]
                        }
                    }
                outfile.write(json.dumps(out_dict) + '\n')
        
create_single_batch_file("products-batch")

#### Initializing client

In [ ]:
from utils.clients import get_client
client = get_client(use_gateway=True)

#### Send Batches to API Server

In [ ]:
# Fetch all the batches

batch_files = glob('../assignment_2/documents/products-batch-*.jsonl')

my_files = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_files.append(batch_input_file)

    

##### Generating Unique ID for job queue

In [ ]:
my_id = f'deploying-ai-{uuid.uuid4().hex}'

#### Creating Batches Server Side

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Canada Statistics Product Description ({my_id}) {timestamp}"

for file in tqdm(my_files):
    client.batches.create(
            input_file_id = file.id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

In [ ]:

batch_processes = client.batches.list().to_dict()
batch_processes

In [ ]:

batch_info= [
    {
    'batch_id': batch['id'],
    'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id'],
    'input_file_id': batch['input_file_id']}
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_complete = [
    
    batch  for batch in batch_info if batch['status'] == 'completed'
]
batch_incomplete = [
    batch  for batch in batch_info if batch['status'] != 'completed'
]

print(f"Batch complete: {len(batch_complete)}\n\nBatch incomplete: {len(batch_incomplete)}")

#### Post-processing Response

In [ ]:


selected = batch_complete[0]

print("status:", selected["status"])
print("output:", selected["output_file_id"])
print("completed_at:", selected.get("completed_at"))
print("counts:", selected.get("request_counts"))

In [ ]:
file_info = client.files.retrieve("file-QnyXFbwQGpAZSN8eTfWCcF")
print(f"File size: {file_info.bytes / 1024 / 1024:.2f} MB")

In [ ]:

def embedding_output_json():

    for index, batch in enumerate(batch_complete):
        response = client.files.content(batch["output_file_id"])
        embedding_lines = [json.loads(line) for line in response.text.splitlines() if line.strip()]

        with open(f"../assignment_2/embeddings/products-batch-{index}.json", 'w') as outfile:
                outfile.write(json.dumps(embedding_lines))
        
embedding_output_json()

In [ ]:
structured_products_embeddings = []

all_embeddings_path = glob('../assignment_2/embeddings/products-batch-*.json')

print(all_embeddings_path)

for path in all_embeddings_path:

    with open(path, "r") as file:
        data = json.load(file)

    structured_products_embeddings += data

print(structured_products_embeddings[:3])

# embedding = ["response"]["body"]["data"][0]["embeddings"]
# product_id = "custom_id"

#### Postprocessing Response

In [ ]:
def get_metadata(product_id):
    metadata_match = [item["metadata"] for item in structured_products if item["productId"] == int(product_id)]
    return metadata_match

def get_content(product_id):
    content_match = [item["metadata"] for item in structured_products if item["productId"] == int(product_id)]
    return content_match
    

In [ ]:
chroma_inputs = []

def create_chroma_inputs():
        
    for embed_item in structured_products_embeddings:

        product_id =  embed_item['custom_id']

        chroma_input = {
            'id': product_id,
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'documents': get_content(product_id),
            'metadata': get_metadata(product_id)
        }

        chroma_inputs.append(chroma_input)

create_chroma_inputs()

### Store in vector database

#### Initiliaze Collection

In [ ]:


COLLECTION_NAME = "product_list"
CHROMA_URL = os.getenv("CHROMA_URL")

def setup_collection(chroma_url:str=CHROMA_URL,
                     collection_name: str = COLLECTION_NAME):
    chroma_client = chromadb.HttpClient(host=chroma_url)
    collections = chroma_client.list_collections()
    if collection_name in [col.name for col in collections]:
        chroma_client.delete_collection(name=collection_name)

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=embedding_function
        
        )
    return collection

setup_collection()


#### Load Inputs

In [ ]:
def load_embeddings_to_db(chroma_inputs:list[dict],
                          collection_name:str,
                          chroma_url:str=CHROMA_URL,
                          batch_size:int=500
                          ):
    collection = setup_collection(chroma_url=chroma_url, collection_name=collection_name)

    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        collection.add(
            documents=[item['documents'] for item in batch],
            embeddings=[item['embedding'] for item in batch],
            metadatas=[item['metadata'] for item in batch],
            ids=[item['id'] for item in batch]
        )


In [62]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

USE_GATEWAY = os.getenv("USE_GATEWAY", "False").lower() == "true"

if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key="any value",
        api_base="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_type="openai",
        model_name=EMBEDDING_MODEL,
        default_headers={
            "x-api-key": os.getenv("API_GATEWAY_KEY")
        }
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name=EMBEDDING_MODEL
    )


In [ ]:
chroma = chromadb.HttpClient(host=CHROMA_URL)
collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)


ValueError: An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: openai vs persisted: default

In [ ]:
collection.query(
    query_texts=["An account of the current population"],
    n_results=3
)